# Convolutional LSTMを用いた動画像予測

---
## 目的
ConvLSTM [1]を構築し，動画像（フレームの時系列）の未来予測を行う仕組みを理解する．

`rnn.ipynb`などのRNN/LSTMは，各時刻の入力・隠れ状態がベクトル（1次元）でした．画像を扱う場合，これを単純にベクトルへ平坦化して入力すると，画像が持つ空間的な構造（近傍の画素同士の関連）が失われてしまいます．ConvLSTM [1] は，LSTM内部の全結合演算を畳み込み演算に置き換えることで，空間構造を保ったまま，時間方向の情報伝播（LSTMの記憶機構）を行えるようにしたネットワークです．本ノートブックでは，Moving MNIST（動画像として動き回るMNIST数字）を用いて，観測した10フレームから続く10フレームを予測する動画像予測を行います．

[1] Xingjian Shi, Zhourong Chen, Hao Wang, Dit-Yan Yeung, Wai-kin Wong, Wang-chun Woo, "Convolutional LSTM Network: A Machine Learning Approach for Precipitation Nowcasting," NIPS, 2015.

<img src="https://github.com/himidev/Lecture/blob/main/13_rnn/07_ConvLSTM/convLSTM.png?raw=true" width = 70%>

## モジュールのインポート

はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import gzip
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.utils.data as data
import torch.optim as optim
from torch import nn
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## MovingMNIST
学習に使用するMoving MNISTデータセット（動くMNIST数字の動画像）をGoogle Driveからダウンロードする．

In [ ]:
if not os.path.isfile('train-images-idx3-ubyte.gz'):
    gdown.download('https://drive.google.com/uc?id=1lhue3SAroqRtLgPbQknRhfMClCR45qik', 'train-images-idx3-ubyte.gz', quiet=True)

## Data loader
Moving Mnistの読み込みを行います．このときバッチ毎にシーケンス数が同じになるように`MovingMNIST`クラス内で処理を行います．

In [ ]:
def load_mnist(root):
    path = os.path.join(root)
    with gzip.open(path, 'rb') as f:
        mnist = np.frombuffer(f.read(), np.uint8, offset=16)
        mnist = mnist.reshape(-1, 28, 28)
    return mnist


def load_fixed_set(root, is_train):
    filename = 'mnist_test_seq.npy'
    path = os.path.join(root, filename)
    dataset = np.load(path)
    dataset = dataset[..., np.newaxis]
    return dataset


class MovingMNIST(data.Dataset):
    def __init__(self, root, is_train, n_frames_input, n_frames_output, num_objects,
                 transform=None):
        super().__init__()

        self.dataset = None
        if is_train:
            self.mnist = load_mnist(root)
        else:
            if num_objects[0] != 2:
                self.mnist = load_mnist(root)
            else:
                self.dataset = load_fixed_set(root, False)
        self.length = int(1e4) if self.dataset is None else self.dataset.shape[1]

        self.is_train = is_train
        self.num_objects = num_objects
        self.n_frames_input = n_frames_input
        self.n_frames_output = n_frames_output
        self.n_frames_total = self.n_frames_input + self.n_frames_output
        self.transform = transform
        self.image_size_ = 64
        self.digit_size_ = 28
        self.step_length_ = 0.1

    def get_random_trajectory(self, seq_length):
        canvas_size = self.image_size_ - self.digit_size_
        x = random.random()
        y = random.random()
        theta = random.random() * 2 * np.pi
        v_y = np.sin(theta)
        v_x = np.cos(theta)

        start_y = np.zeros(seq_length)
        start_x = np.zeros(seq_length)
        for i in range(seq_length):
            y += v_y * self.step_length_
            x += v_x * self.step_length_

            if x <= 0:
                x = 0
                v_x = -v_x
            if x >= 1.0:
                x = 1.0
                v_x = -v_x
            if y <= 0:
                y = 0
                v_y = -v_y
            if y >= 1.0:
                y = 1.0
                v_y = -v_y
            start_y[i] = y
            start_x[i] = x

        start_y = (canvas_size * start_y).astype(np.int32)
        start_x = (canvas_size * start_x).astype(np.int32)
        return start_y, start_x

    def generate_moving_mnist(self, num_digits=2):
        data = np.zeros((self.n_frames_total, self.image_size_, self.image_size_), dtype=np.float32)
        for n in range(num_digits):
            start_y, start_x = self.get_random_trajectory(self.n_frames_total)
            ind = random.randint(0, self.mnist.shape[0] - 1)
            digit_image = self.mnist[ind]
            for i in range(self.n_frames_total):
                top = start_y[i]
                left = start_x[i]
                bottom = top + self.digit_size_
                right = left + self.digit_size_
                data[i, top:bottom, left:right] = np.maximum(data[i, top:bottom, left:right], digit_image)

        data = data[..., np.newaxis]
        return data

    def __getitem__(self, idx):
        length = self.n_frames_input + self.n_frames_output
        if self.is_train or self.num_objects[0] != 2:
            num_digits = random.choice(self.num_objects)
            images = self.generate_moving_mnist(num_digits)
        else:
            images = self.dataset[:, idx, ...]


        r = 1
        w = int(64 / r)
        images = images.reshape((length, w, r, w, r)).transpose(0, 2, 4, 1, 3).reshape((length, r * r, w, w))

        input = images[:self.n_frames_input]
        if self.n_frames_output > 0:
            output = images[self.n_frames_input:length]
        else:
            output = []

        frozen = input[-1]

        output = torch.from_numpy(output / 255.0).contiguous().float()
        input = torch.from_numpy(input / 255.0).contiguous().float()

        out = [idx, output, input, frozen, np.zeros(1)]
        return out

    def __len__(self):
        return self.length

## 学習パラメータの設定やデータの読み込み
 各パラメータを設定します．本実験では20フレームを用い，観測10フレームの動画像を入力し，その後の10フレームの予測動画像を出力します．MovingMNIST内のrootで，driveに置いてあるファイルパスを指定します．このとき，学習用と評価用にsplitしたファイルが各Folderです．


In [ ]:
frame_input = 10
frame_output = 10
batch = 8
lr = 0.001
epochs = 10

trainFolder = MovingMNIST(is_train=True,
                          root='./train-images-idx3-ubyte.gz',
                          n_frames_input=frame_input,
                          n_frames_output=frame_output,
                          num_objects=[3])
validFolder = MovingMNIST(is_train=False,
                          root='./train-images-idx3-ubyte.gz',
                          n_frames_input=frame_input,
                          n_frames_output=frame_output,
                          num_objects=[3])

trainLoader = torch.utils.data.DataLoader(trainFolder,
                                          batch_size=batch,
                                          shuffle=False)
validLoader = torch.utils.data.DataLoader(validFolder,
                                          batch_size=batch,
                                          shuffle=False)

## Convolutional LSTMセル
LSTMの入力層・隠れ層間の全結合演算を，画像の空間構造を保ったまま扱える畳み込み演算に置き換えた`CLSTM_cell`を定義する．各時刻の入力`x`と前時刻の隠れ状態`h`をチャンネル方向に結合し，畳み込み層で入力・忘却・セル・出力の4つのゲートをまとめて計算する点はLSTMと同様である．`inputs=None`の場合は，各時刻の入力としてゼロテンソルを用いる（Decoderで使用）．

In [ ]:
class CLSTM_cell(nn.Module):
    def __init__(self, shape, input_channels, filter_size, num_features):
        super().__init__()
        self.shape = shape  # (H, W)
        self.input_channels = input_channels
        self.num_features = num_features
        padding = (filter_size - 1) // 2
        self.conv = nn.Sequential(
            nn.Conv2d(input_channels + num_features, 4 * num_features, filter_size, stride=1, padding=padding),
            nn.GroupNorm(4 * num_features // 32, 4 * num_features),
        )

    def forward(self, inputs=None, hidden_state=None, seq_len=None):
        if seq_len is None:
            seq_len = inputs.size(0)
        device = self.conv[0].weight.device

        if hidden_state is None:
            batch_size = inputs.size(1)
            hx = torch.zeros(batch_size, self.num_features, *self.shape, device=device)
            cx = torch.zeros(batch_size, self.num_features, *self.shape, device=device)
        else:
            hx, cx = hidden_state

        outputs = []
        for t in range(seq_len):
            if inputs is None:
                x = torch.zeros(hx.size(0), self.input_channels, *self.shape, device=device)
            else:
                x = inputs[t]

            gates = self.conv(torch.cat([x, hx], dim=1))
            in_gate, forget_gate, cell_gate, out_gate = torch.split(gates, self.num_features, dim=1)
            in_gate = torch.sigmoid(in_gate)
            forget_gate = torch.sigmoid(forget_gate)
            cell_gate = torch.tanh(cell_gate)
            out_gate = torch.sigmoid(out_gate)

            cx = forget_gate * cx + in_gate * cell_gate
            hx = out_gate * torch.tanh(cx)
            outputs.append(hx)

        return torch.stack(outputs), (hx, cx)

## Encoder
Moving MNISTの各フレームを畳み込み層で特徴抽出しつつ，`CLSTM_cell`で時間方向の情報を伝播することで，動画像の特徴量を求める．3段のステージ（`stage1`〜`stage3`，`rnn1`〜`rnn3`）を通るごとに空間サイズを64→32→16へ縮小し，チャンネル数を増やしていく．各`CLSTM_cell`の最終的な内部状態（`state1`〜`state3`）がDecoderに渡される．

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn1 = CLSTM_cell(shape=(64, 64), input_channels=16, filter_size=5, num_features=64)

        self.stage2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn2 = CLSTM_cell(shape=(32, 32), input_channels=64, filter_size=5, num_features=96)

        self.stage3 = nn.Sequential(
            nn.Conv2d(96, 96, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.rnn3 = CLSTM_cell(shape=(16, 16), input_channels=96, filter_size=5, num_features=96)

    def forward_by_stage(self, inputs, stage, rnn):
        s, b, c, h, w = inputs.size()
        inputs = stage(inputs.reshape(s * b, c, h, w))
        inputs = inputs.reshape(s, b, inputs.size(1), inputs.size(2), inputs.size(3))
        outputs, state = rnn(inputs, None)
        return outputs, state

    def forward(self, inputs):
        inputs = inputs.transpose(0, 1)  # S, B, C, H, W
        inputs, state1 = self.forward_by_stage(inputs, self.stage1, self.rnn1)
        inputs, state2 = self.forward_by_stage(inputs, self.stage2, self.rnn2)
        _, state3 = self.forward_by_stage(inputs, self.stage3, self.rnn3)
        return state1, state2, state3

## Decoder
Encoderで得られた3つの内部状態（`state1`〜`state3`）から，逆順（`stage3`→`stage1`）に転置畳み込み（`ConvTranspose2d`）と`CLSTM_cell`を適用し，元の画像サイズまで空間解像度を復元しながら`seq_len`フレーム分の予測を生成する．

`rnn.ipynb`や`seq2seq.ipynb`のデコーダとは異なり，本ノートブックのデコーダは各時刻の入力として実際の画像（真値でも自身の予測結果でも）を一切受け取らない．`CLSTM_cell.forward`で`inputs=None`の場合はゼロテンソルが入力される仕組みになっており，Encoderから受け取った内部状態だけを頼りに，内部状態の時間発展のみで予測を生成する．そのため，teacher forcing（真値を入力する評価）と自己回帰的な評価という区別自体が生じず，学習時・評価時とも常に同じ手続きで予測が行われる．

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn3 = CLSTM_cell(shape=(16, 16), input_channels=96, filter_size=5, num_features=96)
        self.stage3 = nn.Sequential(
            nn.ConvTranspose2d(96, 96, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.rnn2 = CLSTM_cell(shape=(32, 32), input_channels=96, filter_size=5, num_features=96)
        self.stage2 = nn.Sequential(
            nn.ConvTranspose2d(96, 96, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.rnn1 = CLSTM_cell(shape=(64, 64), input_channels=96, filter_size=5, num_features=64)
        self.stage1 = nn.Sequential(
            nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(16, 1, kernel_size=1, stride=1, padding=0),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward_by_stage(self, inputs, state, stage, rnn, seq_len):
        inputs, state = rnn(inputs, state, seq_len=seq_len)
        s, b, c, h, w = inputs.size()
        inputs = stage(inputs.reshape(s * b, c, h, w))
        inputs = inputs.reshape(s, b, inputs.size(1), inputs.size(2), inputs.size(3))
        return inputs

    def forward(self, state1, state2, state3, seq_len):
        inputs = self.forward_by_stage(None, state3, self.stage3, self.rnn3, seq_len)
        inputs = self.forward_by_stage(inputs, state2, self.stage2, self.rnn2, seq_len)
        inputs = self.forward_by_stage(inputs, state1, self.stage1, self.rnn1, seq_len)
        return inputs.transpose(0, 1)  # B, S, C, H, W

## Encoder-Decoderへの入力
EncoderとDecoderをまとめて扱う`ED`クラスを定義する．入力動画像をEncoderで内部状態に変換し，その内部状態からDecoderが将来フレームを生成するという一連の処理を，1回の呼び出しでまとめて行えるようにする．

In [ ]:
class ED(nn.Module):
    def __init__(self, encoder, decoder, seq_len):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.seq_len = seq_len

    def forward(self, inputs):
        state1, state2, state3 = self.encoder(inputs)
        outputs = self.decoder(state1, state2, state3, self.seq_len)
        return outputs

## モデル定義
EncoderとDecoderをインスタンス化し，`ED`としてまとめる．損失関数には`MSELoss`を，最適化手法には`Adam`を用いる．

In [ ]:
encoder = Encoder()
decoder = Decoder()
net = ED(encoder, decoder, frame_output).to(device)

lossfunction = nn.MSELoss().to(device)
optimizer = optim.Adam(net.parameters(), lr=lr)

## 学習
学習と検証用データを回します．動画像の特徴を捉えるには数十〜数百エポックの学習が必要で，本設定では学習に3日ほどかかります．演習時間内に学習を終えることは現実的ではないため，以下の学習セルは実行せず，次の「Pre-trained model」セルから学習済みモデルをダウンロードして評価・可視化に進んでください．学習セルは参考用として掲載しています．

In [ ]:
valid_losses = []
for epoch in range(0, epochs + 1):
    ######################
    # 学習
    ######################
    net.train()
    train_loss = 0
    for i, (idx, targetVar, inputVar, _, _) in enumerate(trainLoader):
        inputs = inputVar.to(device)  # B,S,C,H,W
        label = targetVar.to(device)  # B,S,C,H,W
        optimizer.zero_grad()
        pred = net(inputs)  # B,S,C,H,W
        loss = lossfunction(pred, label)
        loss_aver = loss.item() / batch
        train_loss += loss_aver
        loss.backward()
        torch.nn.utils.clip_grad_value_(net.parameters(), clip_value=10.0)
        optimizer.step()
    # モデルを保存
    # torch.save(net.state_dict(), 'pre-train.pth')

    ######################
    # 検証
    ######################
    net.eval()
    test_loss = 0
    with torch.no_grad():
        for i, (idx, targetVar, inputVar, _, _) in enumerate(validLoader):
            inputs = inputVar.to(device)
            label = targetVar.to(device)
            pred = net(inputs)
            loss = lossfunction(pred, label)
            loss_aver = loss.item() / batch
            test_loss += loss_aver
            valid_losses.append(loss_aver)
    print('%03d-epoch: train_loss %.5f : val_loss %.5f' % (epoch, train_loss / len(trainLoader), test_loss / len(validLoader)))
    torch.cuda.empty_cache()

## Pre-trained model
学習時間が長いため，学習済みモデルをダウンロードして使用します．ご自身で学習から始める場合は，上の学習セルのコメントアウトされているモデル保存部分を有効にしてください．

In [ ]:
if not os.path.isfile('pre-train.pth'):
    gdown.download('https://drive.google.com/uc?id=1geXhoW6ij1JodjcOOqHTgYVjdFGbSraI', 'pre-train.pth', quiet=True)

# 学習済み重みは旧実装（レイヤー名にconv1_leaky_1等を用いる構成）で保存されているため，
# 現在のモデル構成（stage1.0など）に合わせてキー名を変換してから読み込む．
key_map = {
    'encoder.stage1.conv1_leaky_1': 'encoder.stage1.0',
    'encoder.stage2.conv2_leaky_1': 'encoder.stage2.0',
    'encoder.stage3.conv3_leaky_1': 'encoder.stage3.0',
    'decoder.stage3.deconv1_leaky_1': 'decoder.stage3.0',
    'decoder.stage2.deconv2_leaky_1': 'decoder.stage2.0',
    'decoder.stage1.conv3_leaky_1': 'decoder.stage1.0',
    'decoder.stage1.conv4_leaky_1': 'decoder.stage1.2',
}
state_dict = torch.load("./pre-train.pth", map_location=device)
for old_prefix, new_prefix in key_map.items():
    for suffix in ['weight', 'bias']:
        old_key = f'{old_prefix}.{suffix}'
        if old_key in state_dict:
            state_dict[f'{new_prefix}.{suffix}'] = state_dict.pop(old_key)
net.load_state_dict(state_dict)

## 評価
予測した動画像と真値の動画像を比較します．損失計算はMSEで計算します．

In [ ]:
net.eval()
test_loss = 0
with torch.no_grad():
    for i, (idx, targetVar, inputVar, _, _) in enumerate(validLoader):
        inputs = inputVar.to(device)
        label = targetVar.to(device)
        pred = net(inputs)
        loss = lossfunction(pred, label)
        loss_aver = loss.item() / batch
        test_loss += loss_aver
print('loss:{:.6f}'.format(test_loss / len(validLoader)))

## 可視化
検証データの先頭バッチを用いて，10フレームの正解画像（`gt`）と予測画像（`pred`）を並べて表示し，予測結果を確認する．

In [ ]:
net.eval()

# 可視化には検証データの最初の1バッチのみを使用する
idx, targetVar, inputVar, _, _ = next(iter(validLoader))
inputs = inputVar.to(device)
label = targetVar.to(device)
with torch.no_grad():
    pred = net(inputs)

plt.figure(figsize=[20, 6])
for frame_idx in range(10):
    plt.subplot(2, 10, frame_idx + 1)
    plt.imshow(label[0, frame_idx, 0].detach().cpu().numpy())
    plt.axis('off')
    plt.title('gt # %2d' % frame_idx)
    plt.subplot(2, 10, frame_idx + 11)
    plt.imshow(pred[0, frame_idx, 0].detach().cpu().numpy())
    plt.axis('off')
    plt.title('pred # %2d' % frame_idx)

## 課題

1. `num_objects`（動画像内の動くMNIST数字の数）を変更して学習済みモデルの予測結果を比較し，動く物体の数が増えると予測がどのように難しくなるか確認してください．
2. 可視化結果を1フレーム目から10フレーム目まで順に見比べ，予測が先の時刻になるほどどのように崩れていくか観察してください．
3. `CLSTM_cell`内の`nn.GroupNorm`を`nn.BatchNorm2d`に置き換えて（学習から行う必要があります），学習の安定性や予測結果にどのような違いが出るか考察してください．